## Imports

In [ ]:
import os, json, yaml
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    roc_auc_score, precision_recall_curve, f1_score, accuracy_score
)
import joblib
sns.set_context('talk'); sns.set_style('whitegrid')


## Paths and Config 

In [ ]:
def detect_root():
    cwd = Path.cwd()
    for p in [cwd, cwd.parent, cwd.parent.parent]:
        if all((p/d).exists() for d in ['config','data','scripts','notebooks']):
            return p
    return cwd

ROOT = detect_root()
CONFIG_PATH = ROOT/'config'/'config.yaml'
SPLITS_DIR  = ROOT/'data'/'splits'
CLASSICAL_DIR = ROOT/'data'/'processed'/'classical'
NEURAL_DIR  = ROOT/'data'/'processed'/'neural'
PLOTS_DIR   = ROOT/'results'/'plots'; PLOTS_DIR.mkdir(parents=True, exist_ok=True)
EVALS_DIR   = ROOT/'results'/'evaluations'; EVALS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = ROOT/'models'/'trained'

cfg = yaml.safe_load(open(CONFIG_PATH,'r'))
dcfg = cfg.get('config',{})
tcfg = cfg.get('training',{})
classes = dcfg.get('classes',[])
num_classes = len(classes)
print("Classes:", classes)
print("Num classes:", num_classes)


### Load Classical Test Data and Models

In [ ]:
def read_manifest_stems(path: Path):
    stems=set()
    if not path.exists(): return stems
    for line in open(path,'r'):
        line=line.strip()
        if not line: continue
        rel,_ = line.split(',',1)
        stems.add(Path(rel).stem)
    return stems

test_stems = read_manifest_stems(SPLITS_DIR/'test.txt')

# Load classical features
X_test_classical, y_test_classical = [], []
name_to_idx = {c:i for i,c in enumerate(classes)}
for c in classes:
    cdir = CLASSICAL_DIR/c
    if not cdir.exists(): continue
    for f in cdir.glob('*.npy'):
        base = f.stem.split('_seg')[0]
        if base in test_stems:
            X_test_classical.append(np.load(f))
            y_test_classical.append(name_to_idx[c])

X_test_classical = np.array(X_test_classical)
y_test_classical = np.array(y_test_classical)
print("Classical test shape:", X_test_classical.shape)

# Load classical models
svm_model = joblib.load(MODELS_DIR/'classical'/'svm_model.pkl')
rf_model = joblib.load(MODELS_DIR/'classical'/'rf_model.pkl')
xgb_model = joblib.load(MODELS_DIR/'classical'/'xgb_model.pkl')
print("Loaded SVM, RF, XGBoost")


### Load Neural Test Data and Models

In [ ]:
from tensorflow.keras.preprocessing.image import smart_resize

def collect_neural_for_split(split_stems, class_names):
    X_list, y_list = [], []
    name_to_idx = {c:i for i,c in enumerate(class_names)}
    for c in class_names:
        cdir = NEURAL_DIR/c
        if not cdir.exists(): continue
        for f in cdir.glob('*.npy'):
            base=f.stem.split('_seg')[0]
            if base in split_stems:
                arr = np.load(f)
                if arr.ndim==3: arr = np.transpose(arr,(1,2,0))
                elif arr.ndim==2: arr = arr[:,:,None]
                else: continue
                X_list.append(arr.astype(np.float32))
                y_list.append(name_to_idx[c])
    if not X_list: return np.empty((0,)), np.empty((0,),dtype=int)
    X = np.stack(X_list, axis=0); y = np.array(y_list, dtype=int)
    return X, y

X_test_neural, y_test_neural = collect_neural_for_split(test_stems, classes)

# Resize to (64, 101) for neural models
X_test_neural = np.array([smart_resize(x, (64, 101)) for x in X_test_neural])
X_test_neural = np.mean(X_test_neural, axis=-1, keepdims=True).astype(np.float32)
X_test_neural = (X_test_neural - X_test_neural.min()) / (X_test_neural.max() - X_test_neural.min() + 1e-7)

print("Neural test shape:", X_test_neural.shape)

# Load neural models
custom_cnn = keras.models.load_model(MODELS_DIR/'neural'/'custom_cnn.h5')
yamnet_ft = keras.models.load_model(MODELS_DIR/'neural'/'yamnet_ft.h5')
mobilenet_ft = keras.models.load_model(MODELS_DIR/'neural'/'mobilenet_ft.h5')
student_distilled = keras.models.load_model(MODELS_DIR/'neural'/'student_cnn_distilled_nb.h5')
print("Loaded custom CNN, YAMNet, MobileNetV2, Distilled Student")


### Evaluate all Models

In [ ]:
results = {}

# Classical models
print("=== Classical Models ===")
for name, model in [('SVM', svm_model), ('Random Forest', rf_model), ('XGBoost', xgb_model)]:
    y_pred = model.predict(X_test_classical)
    acc = accuracy_score(y_test_classical, y_pred)
    f1 = f1_score(y_test_classical, y_pred, average='weighted')
    print(f"{name}: Acc={acc:.4f}, F1={f1:.4f}")
    results[name] = {'accuracy': acc, 'f1': f1, 'y_pred': y_pred}

# Neural models
print("\n=== Neural Models ===")
for name, model in [
    ('Custom CNN', custom_cnn),
    ('YAMNet FT', yamnet_ft),
    ('MobileNet FT', mobilenet_ft),
    ('Student Distilled', student_distilled)
]:
    y_pred_proba = model.predict(X_test_neural, batch_size=32)
    y_pred = np.argmax(y_pred_proba, axis=1)
    acc = accuracy_score(y_test_neural, y_pred)
    f1 = f1_score(y_test_neural, y_pred, average='weighted')
    print(f"{name}: Acc={acc:.4f}, F1={f1:.4f}")
    results[name] = {'accuracy': acc, 'f1': f1, 'y_pred': y_pred, 'y_pred_proba': y_pred_proba}

print("\n=== Summary ===")
df = pd.DataFrame({
    k: {'Accuracy': v['accuracy'], 'F1': v['f1']}
    for k, v in results.items()
}).T
print(df.round(4))


### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    y_pred = res['y_pred']
    y_true = y_test_classical if 'SVM' in name or 'Random Forest' in name or 'XGBoost' in name else y_test_neural
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                xticklabels=classes, yticklabels=classes, cbar=False)
    axes[idx].set_title(name)
    axes[idx].set_ylabel('True'); axes[idx].set_xlabel('Pred')

# Hide unused subplots
for idx in range(len(results), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(PLOTS_DIR/'all_models_confusion.png', dpi=150)
plt.show()


### Classification Reports

In [ ]:
for name, res in results.items():
    y_pred = res['y_pred']
    y_true = y_test_classical if 'SVM' in name or 'Random Forest' in name or 'XGBoost' in name else y_test_neural
    print(f"\n{'='*50}")
    print(f"{name}")
    print('='*50)
    print(classification_report(y_true, y_pred, target_names=classes))
    
    # Save to JSON
    report = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    with open(EVALS_DIR/f'{name.lower().replace(" ", "_")}_report.json', 'w') as f:
        json.dump(report, f, indent=2)


### ROC Curves (Neural Models)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

neural_models = [
    ('Custom CNN', custom_cnn, X_test_neural, y_test_neural),
    ('YAMNet FT', yamnet_ft, X_test_neural, y_test_neural),
    ('MobileNet FT', mobilenet_ft, X_test_neural, y_test_neural),
    ('Student Distilled', student_distilled, X_test_neural, y_test_neural),
]

for idx, (name, model, X_test, y_test) in enumerate(neural_models):
    y_pred_proba = model.predict(X_test, batch_size=32, verbose=0)
    
    # One-vs-Rest ROC
    if num_classes == 2:
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba[:, 1])
        roc_auc = auc(fpr, tpr)
        axes[idx].plot(fpr, tpr, lw=2, label=f'ROC (AUC={roc_auc:.3f})')
    else:
        for i, c in enumerate(classes):
            y_bin = (y_test == i).astype(int)
            fpr, tpr, _ = roc_curve(y_bin, y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            axes[idx].plot(fpr, tpr, lw=1.5, label=f'{c} (AUC={roc_auc:.3f})')
    
    axes[idx].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[idx].set_xlabel('False Positive Rate')
    axes[idx].set_ylabel('True Positive Rate')
    axes[idx].set_title(f'{name} ROC Curves')
    axes[idx].legend(loc='lower right', fontsize=8)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR/'neural_models_roc.png', dpi=150)
plt.show()


### Model Comparison Bar Chart

In [ ]:
df_results = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'F1-Score': [results[m]['f1'] for m in results.keys()],
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
df_results.set_index('Model')[['Accuracy']].plot(kind='barh', ax=ax1, color='skyblue')
ax1.set_xlabel('Accuracy')
ax1.set_title('Model Accuracy Comparison')
ax1.set_xlim([0, 1])
for i, v in enumerate(df_results['Accuracy']):
    ax1.text(v + 0.02, i, f'{v:.3f}', va='center')

# F1-Score
df_results.set_index('Model')[['F1-Score']].plot(kind='barh', ax=ax2, color='lightcoral')
ax2.set_xlabel('F1-Score (weighted)')
ax2.set_title('Model F1-Score Comparison')
ax2.set_xlim([0, 1])
for i, v in enumerate(df_results['F1-Score']):
    ax2.text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig(PLOTS_DIR/'model_comparison.png', dpi=150)
plt.show()

print(df_results.to_string(index=False))
